# Symptom Text QNN — Google Colab Reproduction

This notebook recreates the complete project in a Colab runtime: upload the CSV, validate/deduplicate it, run the tests, compare classical baselines, tune the 19 QNN configurations on validation data, evaluate the selected circuits on five held-out splits, generate plots, and run a sample prediction.

**Educational project only — not a medical tool.** Text features are classical TF-IDF and SVD. The quantum circuits run on Qiskit Aer’s simulator. Classical baselines substantially outperform the QNNs in this dataset.

> **Runtime note:** Full Phase A and five-split QNN evaluation take a long time. Keep the Colab tab active and save/download results before the runtime expires. A Colab GPU does not automatically accelerate these Aer statevector runs.


In [ ]:
%pip -q install "numpy<3" "scipy>=1.11" "pandas>=2.1" "scikit-learn>=1.4,<2" "qiskit>=1.2,<3" "qiskit-aer>=0.15,<1" "matplotlib>=3.8" "seaborn>=0.13" "joblib>=1.3" "pytest>=8"


## 1. Set up the project code

In [ ]:
project_sources = {
    'src/__init__.py': '"""Symptom text quantum classifier study."""\n',
    'src/data.py': '"""Leakage-safe data loading and classical text feature pipeline."""\nfrom dataclasses import dataclass\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom sklearn.decomposition import TruncatedSVD\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import MinMaxScaler, StandardScaler\n\nLABELS = ["Common Cold", "allergy", "Pneumonia", "Bronchial Asthma", "Dengue", "Malaria", "Typhoid", "Chicken pox"]\nLABELS = ["Common Cold", "Allergy", "Pneumonia", "Bronchial Asthma", "Dengue", "Malaria", "Typhoid", "Chicken pox"]\nLABEL_TO_ID = {name: i for i, name in enumerate(LABELS)}\nDEFAULT_DATA = Path(__file__).resolve().parents[1] / "data" / "Symptom2Disease.csv"\n\n@dataclass\nclass DatasetSplit:\n    X_train: np.ndarray\n    X_test: np.ndarray\n    y_train: np.ndarray\n    y_test: np.ndarray\n    X_train_pca: np.ndarray\n    X_test_pca: np.ndarray\n    train_text: np.ndarray\n    test_text: np.ndarray\n    vectorizer: TfidfVectorizer\n    svd: TruncatedSVD\n    angle_scaler: MinMaxScaler\n    pca_scaler: StandardScaler\n\n\ndef load_frame(path=DEFAULT_DATA):\n    df = pd.read_csv(path)\n    if not {"label", "text"}.issubset(df.columns):\n        raise ValueError("CSV must contain \'label\' and \'text\' columns")\n    df = df[["label", "text"]].dropna().copy()\n    df["label"] = df.label.astype(str).str.strip()\n    df["text"] = df.text.astype(str).str.strip()\n    # Dataset uses lower-case \'allergy\'; normalize only this known spelling.\n    df["label"] = df.label.map(lambda s: "Allergy" if s.lower() == "allergy" else s)\n    df = df[df.label.isin(LABELS)].drop_duplicates(subset=["label", "text"]).reset_index(drop=True)\n    return df\n\n\ndef make_split(seed=0, path=DEFAULT_DATA, test_size=.25):\n    df = load_frame(path)\n    tr, te = train_test_split(np.arange(len(df)), test_size=test_size, random_state=seed,\n                              stratify=df.label)\n    train, test = df.iloc[tr], df.iloc[te]\n    vec = TfidfVectorizer(stop_words="english", max_features=None)\n    A = vec.fit_transform(train.text)\n    B = vec.transform(test.text)\n    k = min(8, A.shape[1] - 1, A.shape[0] - 1)\n    svd = TruncatedSVD(n_components=k, n_iter=7, random_state=seed)\n    U, V = svd.fit_transform(A), svd.transform(B)\n    # Pad to 8 if tiny/custom input yields fewer components.\n    U = np.pad(U, ((0,0),(0,8-k))); V = np.pad(V, ((0,0),(0,8-k)))\n    angle_scaler = MinMaxScaler(feature_range=(0, np.pi)).fit(U)\n    pca_scaler = StandardScaler().fit(U)\n    return DatasetSplit(angle_scaler.transform(U), angle_scaler.transform(V),\n        train.label.map(LABEL_TO_ID).to_numpy(), test.label.map(LABEL_TO_ID).to_numpy(),\n        pca_scaler.transform(U), pca_scaler.transform(V), train.text.to_numpy(), test.text.to_numpy(),\n        vec, svd, angle_scaler, pca_scaler)\n\n\ndef validation_indices(y, seed=0, fraction=.2):\n    fit, val = train_test_split(np.arange(len(y)), test_size=fraction, random_state=seed, stratify=y)\n    return fit, val\n\n\ndef nested_train_validation(split, seed=0, fraction=.2, view="angle"):\n    """Build a strict fit/validation view with every text transform fit on fit rows only."""\n    from sklearn.model_selection import train_test_split\n    fit, val = train_test_split(np.arange(len(split.y_train)), test_size=fraction,\n                                random_state=seed, stratify=split.y_train)\n    vec = TfidfVectorizer(stop_words="english")\n    a = vec.fit_transform(split.train_text[fit]); b = vec.transform(split.train_text[val])\n    k = min(8, a.shape[1]-1, a.shape[0]-1)\n    svd = TruncatedSVD(n_components=k, n_iter=7, random_state=seed)\n    za, zb = svd.fit_transform(a), svd.transform(b)\n    za=np.pad(za,((0,0),(0,8-k))); zb=np.pad(zb,((0,0),(0,8-k)))\n    scaler = MinMaxScaler((0,np.pi)) if view=="angle" else StandardScaler()\n    xa=scaler.fit_transform(za); xb=scaler.transform(zb)\n    return xa, xb, split.y_train[fit], split.y_train[val], fit, val\n',
    'src/encodings.py': '"""Quantum feature maps; all returned circuits act on an initialized zero state."""\nimport numpy as np\nfrom qiskit import QuantumCircuit\n\n\ndef angle_encoding(x, bandwidth=1.0):\n    x = np.asarray(x, dtype=float)\n    qc = QuantumCircuit(len(x))\n    for i, value in enumerate(x): qc.ry(float(bandwidth * value), i)\n    return qc\n\n\ndef zz_encoding(x, bandwidth=1.0, reps=1):\n    """Havlicek-style ZZ feature map with explicit pairwise interactions."""\n    x = np.asarray(x, dtype=float); n = len(x)\n    qc = QuantumCircuit(n)\n    for _ in range(reps):\n        for i in range(n): qc.h(i)\n        for i in range(n): qc.p(float(2 * bandwidth * x[i]), i)\n        for i in range(n):\n            for j in range(i + 1, n):\n                phase = 2 * (np.pi - bandwidth * x[i]) * (np.pi - bandwidth * x[j])\n                qc.cx(i, j); qc.p(float(phase), j); qc.cx(i, j)\n    return qc\n\n\ndef amplitude_encoding(x):\n    """Prepare normalized real vector in Qiskit\'s little-endian basis ordering."""\n    x = np.asarray(x, dtype=float)\n    if x.shape != (8,): raise ValueError("Amplitude encoding expects exactly 8 features")\n    norm = np.linalg.norm(x)\n    if norm == 0: raise ValueError("Cannot amplitude-encode a zero vector")\n    a = x / norm\n    qc = QuantumCircuit(3)\n    # First split on the most significant qubit (q2), then q1, then q0.\n    # Each prefix denotes fixed values of already prepared higher qubits.\n    for level, target in enumerate((2, 1, 0)):\n        ncontrols = level\n        for prefix in range(2**ncontrols):\n            # Gather amplitudes whose higher-order bits match this prefix.\n            indices = [i for i in range(8) if (i >> (3-level)) == prefix] if level else list(range(8))\n            if level == 0:\n                left = np.linalg.norm(a[:4]); right = np.linalg.norm(a[4:])\n            else:\n                mask = (1 << (3-level)) - 1\n                left = np.linalg.norm([a[i] for i in indices if not ((i >> target) & 1)])\n                right = np.linalg.norm([a[i] for i in indices if (i >> target) & 1])\n            theta = 2 * np.arctan2(right, left)\n            if not ncontrols:\n                qc.ry(float(theta), target)\n            else:\n                controls = list(range(target+1, 3))\n                bits = [(prefix >> (bit-(target+1))) & 1 for bit in controls]\n                for q, bit in zip(controls, bits):\n                    if not bit: qc.x(q)\n                qc.mcry(float(theta), controls, target, None, mode="noancilla")\n                for q, bit in zip(controls, bits):\n                    if not bit: qc.x(q)\n    return qc\n\n\ndef build_encoding(name, x, bandwidth=1.0, reps=1):\n    if name == "angle": return angle_encoding(x, bandwidth)\n    if name == "zz": return zz_encoding(x, bandwidth, reps)\n    if name == "amplitude": return amplitude_encoding(x)\n    raise ValueError(f"Unknown encoding: {name}")\n',
    'src/ansatz.py': '"""Trainable RY/RZ layers with a nearest-neighbor CNOT ring."""\nimport numpy as np\nfrom qiskit import QuantumCircuit\nfrom qiskit.circuit import ParameterVector\n\n\ndef weight_count(n_qubits, depth): return 2 * n_qubits * (depth + 1)\n\n\ndef make_ansatz(n_qubits, depth=1, params=None):\n    if depth < 0: raise ValueError("depth must be nonnegative")\n    params = params or ParameterVector("w", weight_count(n_qubits, depth))\n    if len(params) != weight_count(n_qubits, depth): raise ValueError("incorrect parameter count")\n    qc = QuantumCircuit(n_qubits); p = 0\n    for layer in range(depth + 1):\n        for q in range(n_qubits):\n            qc.ry(params[p], q); p += 1\n            qc.rz(params[p], q); p += 1\n        if layer < depth:\n            for q in range(n_qubits): qc.cx(q, (q + 1) % n_qubits)\n    return qc, params\n',
    'src/qnn.py': '"""Exact-probability variational quantum classifier using Qiskit Aer."""\nfrom dataclasses import dataclass\nimport numpy as np\nfrom qiskit import QuantumCircuit, transpile\nfrom qiskit.circuit import ParameterVector\nfrom qiskit_aer import AerSimulator\nfrom .ansatz import make_ansatz, weight_count\nfrom .encodings import build_encoding\n\n@dataclass\nclass QNNConfig:\n    encoding: str = "angle"\n    depth: int = 1\n    bandwidth: float = 1.0\n    reupload: bool = False\n    encoding_reps: int = 1\n\nclass VariationalClassifier:\n    def __init__(self, config=None, seed=0):\n        self.config = config or QNNConfig()\n        self.n_qubits = 3 if self.config.encoding == "amplitude" else 8\n        self.n_weights = weight_count(self.n_qubits, self.config.depth)\n        self.parameters = ParameterVector("w", self.n_weights)\n        self.weights = np.random.default_rng(seed).normal(0, .15, self.n_weights)\n        self.simulator = AerSimulator(method="statevector", max_parallel_experiments=4, max_parallel_threads=2)\n        self._compiled_cache = {}\n\n    def _circuit(self, x):\n        n = self.n_qubits\n        qc = QuantumCircuit(n)\n        enc = build_encoding(self.config.encoding, x, self.config.bandwidth, self.config.encoding_reps)\n        if not self.config.reupload or self.config.encoding == "amplitude":\n            qc.compose(enc, inplace=True)\n            ansatz, _ = make_ansatz(n, self.config.depth, self.parameters)\n            qc.compose(ansatz, inplace=True)\n        else:\n            # Re-encode the same classical feature vector before every rotation layer.\n            p = 0\n            for layer in range(self.config.depth + 1):\n                qc.compose(enc, inplace=True)\n                for q in range(n):\n                    qc.ry(self.parameters[p], q); p += 1\n                    qc.rz(self.parameters[p], q); p += 1\n                if layer < self.config.depth:\n                    for q in range(n): qc.cx(q, (q + 1) % n)\n        return qc\n\n    def _compiled_circuits(self, X):\n        rows = [np.asarray(x, dtype=float) for x in X]\n        keys = [tuple(row.tolist()) for row in rows]\n        missing = list(dict.fromkeys(key for key in keys if key not in self._compiled_cache))\n        if missing:\n            raw = [self._circuit(np.asarray(key)) for key in missing]\n            compiled = transpile(raw, basis_gates=["u", "cx"], optimization_level=0)\n            for key, circuit in zip(missing, compiled):\n                circuit.save_probabilities(list(range(3)))\n                self._compiled_cache[key] = circuit\n        return [self._compiled_cache[key] for key in keys]\n\n    def _run_with_weights(self, circuits, weights):\n        mapping = {p: float(w) for p,w in zip(self.parameters,weights)}\n        bound = [circuit.assign_parameters(mapping) for circuit in circuits]\n        result = self.simulator.run(bound).result()\n        out = np.asarray([result.data(i)["probabilities"] for i in range(len(bound))], dtype=float)\n        out = np.clip(out, 1e-12, 1)\n        return out / out.sum(axis=1, keepdims=True)\n\n    def probabilities(self, X, weights=None):\n        X = np.atleast_2d(np.asarray(X, dtype=float)); weights = self.weights if weights is None else np.asarray(weights)\n        if weights.shape != (self.n_weights,): raise ValueError("weight vector has wrong shape")\n        return self._run_with_weights(self._compiled_circuits(X), weights)\n\n    def loss(self, X, y, weights=None):\n        p = self.probabilities(X, weights)\n        return float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean())\n\n    def gradient(self, X, y, weights=None):\n        """Parameter-shift gradient; shifted sample circuits are batched in Aer."""\n        w = self.weights.copy() if weights is None else np.asarray(weights, dtype=float).copy()\n        X = np.atleast_2d(np.asarray(X, dtype=float)); y = np.asarray(y)\n        circuits = self._compiled_circuits(X)\n        p0 = self._run_with_weights(circuits, w)\n        shifted_circuits=[]\n        for j in range(len(w)):\n            for sign in (1,-1):\n                shifted = w.copy(); shifted[j] += sign*np.pi/2\n                mapping = {p: float(value) for p,value in zip(self.parameters,shifted)}\n                shifted_circuits.extend(c.assign_parameters(mapping) for c in circuits)\n        result = self.simulator.run(shifted_circuits).result()\n        shifted_probs=np.asarray([result.data(i)["probabilities"] for i in range(len(shifted_circuits))])\n        shifted_probs=shifted_probs.reshape(len(w),2,len(X),8)\n        grad=np.empty_like(w)\n        rows=np.arange(len(y))\n        p_correct=np.clip(p0[rows,y],1e-12,None)\n        for j in range(len(w)):\n            dp=(shifted_probs[j,0,rows,y]-shifted_probs[j,1,rows,y])/2\n            grad[j]=(-dp/p_correct).mean()\n        return grad\n\n    def fit(self, X, y, X_val=None, y_val=None, epochs=20, batch_size=32, lr=.03, seed=0, patience=5):\n        X = np.asarray(X); y = np.asarray(y); rng = np.random.default_rng(seed)\n        m = np.zeros_like(self.weights); v = np.zeros_like(self.weights); step = 0\n        best = self.weights.copy(); best_acc = -1; stale = 0; history=[]\n        for epoch in range(epochs):\n            order = rng.permutation(len(y))\n            for start in range(0, len(y), batch_size):\n                ix = order[start:start+batch_size]; g = self.gradient(X[ix], y[ix])\n                step += 1; m=.9*m+.1*g; v=.999*v+.001*g*g\n                self.weights -= lr*(m/(1-.9**step))/(np.sqrt(v/(1-.999**step))+1e-8)\n            train_loss = self.loss(X, y)\n            if X_val is not None:\n                pred = self.predict(X_val); acc = float(np.mean(pred == y_val))\n            else: acc = -train_loss\n            history.append({"epoch": epoch+1, "loss": train_loss, "validation_accuracy": acc})\n            if acc > best_acc:\n                best_acc=acc; best=self.weights.copy(); stale=0\n            else:\n                stale += 1\n                if stale >= patience: break\n        self.weights = best\n        return history\n\n    def predict_proba(self, X): return self.probabilities(X)\n    def predict(self, X): return self.predict_proba(X).argmax(axis=1)\n',
    'src/classical.py': '"""Classical reference models on the same eight-dimensional features."""\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import SVC, LinearSVC\nfrom sklearn.neural_network import MLPClassifier\n\n\ndef make_model(name, seed=0):\n    if name == "logistic": return LogisticRegression(max_iter=2000, random_state=seed)\n    if name == "linear_svm": return LinearSVC(C=1.0, random_state=seed)\n    if name == "rbf_svm": return SVC(kernel="rbf", C=10.0, gamma="scale", random_state=seed)\n    if name == "mlp": return MLPClassifier(hidden_layer_sizes=(3,), max_iter=2000, learning_rate_init=.01, random_state=seed, early_stopping=False)\n    if name == "tfidf_logistic": return LogisticRegression(max_iter=2000, random_state=seed)\n    raise ValueError(name)\n\n\ndef parameter_count(model):\n    if isinstance(model, SVC): return None  # nonlinear kernel has no finite weight count\n    if hasattr(model, "coef_"):\n        return int(model.coef_.size + model.intercept_.size)\n    return int(sum(w.size for w in model.coefs_) + sum(b.size for b in model.intercepts_))\n\n\ndef evaluate_classical(seed=0, path=None):\n    from .data import make_split, load_frame\n    d = make_split(seed, path) if path else make_split(seed)\n    results=[]\n    for name in ("logistic", "linear_svm", "mlp", "rbf_svm"):\n        model=make_model(name, seed).fit(d.X_train, d.y_train)\n        results.append({"model":name,"accuracy":float(model.score(d.X_test,d.y_test)),"parameters":parameter_count(model)})\n    # Uncompressed TF-IDF baseline, still fit on training data only.\n    frame=load_frame(path) if path else load_frame()\n    from sklearn.model_selection import train_test_split\n    from sklearn.feature_extraction.text import TfidfVectorizer\n    import pandas as pd\n    tr,te=train_test_split(np.arange(len(frame)), test_size=.25, random_state=seed, stratify=frame.label)\n    vec=TfidfVectorizer(stop_words="english"); a=vec.fit_transform(frame.text.iloc[tr]); b=vec.transform(frame.text.iloc[te])\n    from .data import LABEL_TO_ID\n    ytr=frame.label.iloc[tr].map(LABEL_TO_ID).to_numpy(); yte=frame.label.iloc[te].map(LABEL_TO_ID).to_numpy()\n    model=make_model("tfidf_logistic",seed).fit(a,ytr)\n    results.append({"model":"tfidf_logistic","accuracy":float(model.score(b,yte)),"parameters":int(model.coef_.size+model.intercept_.size)})\n    return results\n',
    'src/tuning.py': '"""Validation-only search over circuit structure (19 configurations)."""\nfrom dataclasses import asdict\nimport itertools, json\nfrom pathlib import Path\nimport numpy as np\nfrom .data import make_split, nested_train_validation\nfrom .qnn import QNNConfig, VariationalClassifier\n\nRESULTS=Path(__file__).resolve().parents[1]/"results"\n\ndef phase_a(seeds=(0,), epochs=5, output=RESULTS/"tuning_A.json", path=None):\n    output=Path(output)\n    records=json.loads(output.read_text()) if output.exists() else []\n    completed={(r["seed"],json.dumps(r["config"],sort_keys=True)) for r in records}\n    configs=[]\n    for depth,reup,bw in itertools.product((1,3),(False,True),(1.0,.5)):\n        configs.append(QNNConfig("angle",depth,bw,reup))\n        configs.append(QNNConfig("zz",depth,bw,reup))\n    configs.extend(QNNConfig("amplitude",d,1.0,False) for d in (1,2,3))\n    for seed in seeds:\n        d=make_split(seed,path)\n        for cfg in configs:\n            if (seed,json.dumps(asdict(cfg),sort_keys=True)) in completed:\n                continue\n            xa,xb,ya,yb,_,_=nested_train_validation(d,seed,view="pca" if cfg.encoding=="amplitude" else "angle")\n            model=VariationalClassifier(cfg,seed)\n            hist=model.fit(xa,ya,xb,yb,epochs=epochs,seed=seed)\n            record={"seed":seed,"config":asdict(cfg),"validation_accuracy":max(r["validation_accuracy"] for r in hist),"history":hist}\n            records.append(record)\n            output.parent.mkdir(exist_ok=True,parents=True)\n            output.write_text(json.dumps(records,indent=2))\n            completed.add((seed,json.dumps(asdict(cfg),sort_keys=True)))\n    return records\n',
    'src/runner.py': '"""Experiment runner: classical references, phase A, final held-out evaluation, plots."""\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom sklearn.metrics import accuracy_score, confusion_matrix\nfrom .data import LABELS, make_split, validation_indices, nested_train_validation\nfrom .classical import evaluate_classical\nfrom .qnn import QNNConfig, VariationalClassifier\n\nROOT=Path(__file__).resolve().parents[1]; RESULTS=ROOT/"results"\n\ndef run_classical(seeds=range(5), path=None):\n    rows=[]\n    for seed in seeds:\n        for row in evaluate_classical(seed, path): rows.append({"seed":seed,**row})\n    return rows\n\ndef tune_phase_a(*args,**kwargs):\n    from .tuning import phase_a\n    return phase_a(*args,**kwargs)\n\ndef select_best(path=RESULTS/"tuning_A.json"):\n    records=json.loads(Path(path).read_text())\n    scores={}\n    for r in records:\n        key=(r["config"]["encoding"],json.dumps(r["config"],sort_keys=True))\n        scores.setdefault(key,[]).append(r["validation_accuracy"])\n    by_encoding={}\n    for (enc,cfg),vals in scores.items():\n        if enc not in by_encoding or np.mean(vals)>by_encoding[enc][0]: by_encoding[enc]=(float(np.mean(vals)),json.loads(cfg))\n    return {k:v[1] for k,v in by_encoding.items()}\n\ndef final_evaluation(best=None, seeds=range(5), epochs=5, path=None):\n    best=best or select_best()\n    RESULTS.mkdir(exist_ok=True)\n    output=RESULTS/"final_evaluation.json"\n    rows=json.loads(output.read_text()) if output.exists() else []\n    completed={(r["seed"],r["encoding"]) for r in rows}\n    all_cm={}\n    for row in rows:\n        all_cm.setdefault(row["encoding"],[]).append(confusion_matrix(row["y_true"],row["y_pred"],labels=range(8)))\n    for seed in seeds:\n        d=make_split(seed,path)\n        for enc,cfgdict in best.items():\n            if (seed,enc) in completed: continue\n            cfg=QNNConfig(**cfgdict); xtr=d.X_train_pca if enc=="amplitude" else d.X_train; xte=d.X_test_pca if enc=="amplitude" else d.X_test\n            # Fixed Phase-A epoch budget. Fit on all training rows, then make one test pass.\n            final=VariationalClassifier(cfg,seed)\n            hist=final.fit(xtr,d.y_train,epochs=epochs,seed=seed,patience=epochs+1)\n            proba=final.predict_proba(xte); pred=proba.argmax(axis=1)\n            row={"seed":seed,"encoding":enc,"accuracy":float(accuracy_score(d.y_test,pred)),"config":cfgdict,"y_true":d.y_test.tolist(),"y_pred":pred.tolist(),"probabilities":proba.tolist(),"history":hist}\n            rows.append(row); all_cm.setdefault(enc,[]).append(confusion_matrix(d.y_test,pred,labels=range(8)))\n            output.write_text(json.dumps(rows,indent=2))\n            completed.add((seed,enc))\n    make_plots(rows,all_cm)\n    summary={}\n    for enc in best:\n        scores=[r["accuracy"] for r in rows if r["encoding"]==enc]\n        summary[enc]={"mean_accuracy":float(np.mean(scores)),"std_accuracy":float(np.std(scores,ddof=1)),"n_splits":len(scores),"config":best[enc]}\n    (RESULTS/"qnn_summary.json").write_text(json.dumps(summary,indent=2))\n    return rows\n\ndef make_plots(rows, matrices):\n    import matplotlib.pyplot as plt\n    import seaborn as sns\n    RESULTS.mkdir(exist_ok=True)\n    # Training curves (recorded in validation history).\n    fig,ax=plt.subplots(figsize=(8,5))\n    first={}\n    for r in rows: first.setdefault(r["encoding"],r)\n    for enc,r in first.items(): ax.plot([h["epoch"] for h in r["history"]],[h["loss"] for h in r["history"]],label=enc)\n    ax.set(xlabel="Epoch",ylabel="Training cross-entropy",title="QNN training curves"); ax.legend(); fig.tight_layout(); fig.savefig(RESULTS/"training_curves.png",dpi=160); plt.close(fig)\n    fig,ax=plt.subplots(figsize=(7,4)); encs=sorted({r["encoding"] for r in rows})\n    vals=[[r["accuracy"] for r in rows if r["encoding"]==e] for e in encs]\n    ax.boxplot(vals,tick_labels=encs); ax.set(ylabel="Test accuracy",title="Held-out accuracy by encoding"); fig.tight_layout(); fig.savefig(RESULTS/"accuracy_by_encoding.png",dpi=160); plt.close(fig)\n    depths=sorted({r["config"]["depth"] for r in rows})\n    fig,ax=plt.subplots(figsize=(7,4))\n    vals=[[r["accuracy"] for r in rows if r["config"]["depth"]==d] for d in depths]\n    ax.boxplot(vals,tick_labels=[str(d) for d in depths]); ax.set(xlabel="Ansatz depth",ylabel="Test accuracy",title="Held-out accuracy by depth")\n    fig.tight_layout(); fig.savefig(RESULTS/"accuracy_by_depth.png",dpi=160); plt.close(fig)\n    for enc,ms in matrices.items():\n        fig,ax=plt.subplots(figsize=(9,7)); sns.heatmap(np.sum(ms,axis=0),annot=True,fmt="d",xticklabels=LABELS,yticklabels=LABELS,cmap="Blues",ax=ax)\n        ax.set(xlabel="Predicted",ylabel="True",title=f"Confusion matrix: {enc}"); plt.xticks(rotation=45,ha="right"); fig.tight_layout(); fig.savefig(RESULTS/f"confusion_{enc}.png",dpi=160); plt.close(fig)\n\ndef main():\n    RESULTS.mkdir(exist_ok=True)\n    rows=run_classical()\n    (RESULTS/"classical.json").write_text(json.dumps(rows,indent=2))\n    summary={}\n    for name in sorted({r["model"] for r in rows}):\n        scores=[r["accuracy"] for r in rows if r["model"]==name]\n        params=[r["parameters"] for r in rows if r["model"]==name]\n        known=[p for p in params if p is not None]\n        summary[name]={"accuracy_mean":float(np.mean(scores)),"accuracy_std":float(np.std(scores,ddof=1)),"parameters_mean":float(np.mean(known)) if known else None}\n    (RESULTS/"classical_summary.json").write_text(json.dumps(summary,indent=2))\n    print("Saved classical baselines. Run phase A with: python -m src.runner tune")\n\ndef save_demo_model(config=None, seed=0, epochs=5, path=None):\n    """Fit selected circuit on all labelled examples and persist inference pipeline."""\n    import joblib\n    from .data import load_frame, LABEL_TO_ID\n    from sklearn.feature_extraction.text import TfidfVectorizer\n    from sklearn.decomposition import TruncatedSVD\n    from sklearn.preprocessing import MinMaxScaler, StandardScaler\n    frame=load_frame(path) if path else load_frame(); y=frame.label.map(LABEL_TO_ID).to_numpy()\n    vec=TfidfVectorizer(stop_words="english"); tf=vec.fit_transform(frame.text)\n    svd=TruncatedSVD(n_components=8,random_state=seed); z=svd.fit_transform(tf)\n    if config is None:\n        tuning=json.loads((RESULTS/"tuning_A.json").read_text())\n        config=QNNConfig(**max(tuning,key=lambda r:r["validation_accuracy"])["config"])\n    scaler=StandardScaler() if config.encoding=="amplitude" else MinMaxScaler((0,np.pi))\n    x=scaler.fit_transform(z)\n    model=VariationalClassifier(config,seed); model.fit(x,y,epochs=epochs,seed=seed,patience=epochs+1)\n    RESULTS.mkdir(exist_ok=True)\n    joblib.dump({"vectorizer":vec,"svd":svd,"scaler":scaler,"config":config.__dict__,"weights":model.weights},RESULTS/"demo_model.joblib")\n    return RESULTS/"demo_model.joblib"\n\nif __name__=="__main__":\n    import sys\n    if len(sys.argv)>1 and sys.argv[1]=="tune": print(json.dumps(tune_phase_a(),indent=2))\n    elif len(sys.argv)>1 and sys.argv[1]=="final": print(json.dumps(final_evaluation(),indent=2))\n    elif len(sys.argv)>1 and sys.argv[1]=="demo-model": print(save_demo_model())\n    else: main()\n',
    'src/predict.py': '"""Interactive symptom sentence demo backed by saved preprocessing and QNN weights."""\nimport argparse, json\nfrom pathlib import Path\nimport joblib\nimport numpy as np\nfrom .data import LABELS\nfrom .qnn import QNNConfig, VariationalClassifier\n\nROOT=Path(__file__).resolve().parents[1]\n\ndef predict(sentence, artifact=ROOT/"results"/"demo_model.joblib"):\n    obj=joblib.load(artifact)\n    x=obj["vectorizer"].transform([sentence]); z=obj["svd"].transform(x)\n    model=obj["scaler"].transform(z)\n    if model.shape[1] < 8: model=np.pad(model,((0,0),(0,8-model.shape[1])))\n    qnn=VariationalClassifier(QNNConfig(**obj["config"])); qnn.weights=np.asarray(obj["weights"])\n    probs=qnn.predict_proba(model)[0]\n    return LABELS[int(probs.argmax())], {LABELS[i]:float(probs[i]) for i in range(8)}\n\ndef main():\n    p=argparse.ArgumentParser(description="Experimental symptom text classifier (not a medical tool)")\n    p.add_argument("sentence",nargs="?",help="symptoms in plain English")\n    p.add_argument("--json",action="store_true",help="print a machine-readable result for the web demo")\n    a=p.parse_args(); sentence=a.sentence or input("Describe symptoms: ")\n    label,probs=predict(sentence)\n    if a.json:\n        print(json.dumps({"predicted":label,"probabilities":probs}))\n        return\n    print(f"Predicted dataset label: {label}")\n    for disease,prob in sorted(probs.items(),key=lambda kv:-kv[1]): print(f"{disease:20s} {prob:.3f}")\n\nif __name__=="__main__": main()\n',
    'tests/test_data.py': 'from src.data import LABELS, load_frame, make_split\n\ndef test_dataset_deduplicated_and_expected_size():\n    df=load_frame()\n    assert len(df)==388\n    assert set(df.label)==set(LABELS)\n    assert not df.duplicated(["label","text"]).any()\n\ndef test_split_is_stratified_seeded_and_disjoint():\n    a=make_split(2); b=make_split(2)\n    assert (a.y_train==b.y_train).all() and (a.y_test==b.y_test).all()\n    assert len(a.y_train)==291 and len(a.y_test)==97\n    assert set(a.y_train)==set(range(8)) and set(a.y_test)==set(range(8))\n    assert a.X_train.shape==(291,8) and a.X_test.shape==(97,8)\n    assert (a.X_train.min()>=0) and (a.X_train.max()<=3.141593)\n',
    'tests/test_circuits.py': 'import numpy as np\nfrom qiskit.quantum_info import Statevector\nfrom src.ansatz import make_ansatz, weight_count\nfrom src.encodings import amplitude_encoding, angle_encoding, zz_encoding\nfrom src.qnn import QNNConfig, VariationalClassifier\n\ndef test_amplitude_state_matches_normalized_vector():\n    x=np.array([1.,2.,3.,4.,5.,6.,7.,8.]); qc=amplitude_encoding(x)\n    actual=Statevector.from_instruction(qc).data\n    assert np.allclose(actual,x/np.linalg.norm(x),atol=1e-8)\n\ndef test_ansatz_weight_count_and_unique_use():\n    for n,d in ((8,1),(3,2)):\n        qc,p=make_ansatz(n,d)\n        assert len(p)==2*n*(d+1)==weight_count(n,d)\n        assert len(qc.parameters)==len(p)\n\ndef test_readout_probabilities_match_statevector():\n    for encoding, x in (("angle",np.linspace(.1,2.1,8)),("amplitude",np.linspace(-1,1,8))):\n        model=VariationalClassifier(QNNConfig(encoding,1),seed=3); observed=model.probabilities([x])[0]\n        qc=model._circuit(x).assign_parameters(dict(zip(model.parameters,model.weights)))\n        qc.data = [item for item in qc.data if item.operation.name != "save_probabilities"]\n        expected=Statevector.from_instruction(qc).probabilities(qargs=[0,1,2])\n        assert np.allclose(observed,expected,atol=1e-8)\n        assert np.isclose(observed.sum(),1)\n\ndef test_parameter_shift_gradient_matches_finite_difference():\n    model=VariationalClassifier(QNNConfig("angle",0),seed=4)\n    X=np.array([[.1]*8,[.7]*8]); y=np.array([0,1]); w=model.weights.copy()\n    grad=model.gradient(X,y,w)\n    eps=1e-5; fd=[]\n    for j in range(len(w)):\n        plus=w.copy(); minus=w.copy(); plus[j]+=eps; minus[j]-=eps\n        fd.append((model.loss(X,y,plus)-model.loss(X,y,minus))/(2*eps))\n    assert np.allclose(grad,fd,atol=2e-4,rtol=2e-3)\n',
    'pyproject.toml': '[build-system]\nrequires = ["setuptools>=68"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "symptom-qnn"\nversion = "0.1.0"\ndescription = "A transparent quantum-classifier study for symptom text"\nrequires-python = ">=3.10"\ndependencies = [\n  "numpy>=1.26,<3", "scipy>=1.11", "pandas>=2.1", "scikit-learn>=1.4",\n  "qiskit>=1.2,<3", "qiskit-aer>=0.15,<1", "matplotlib>=3.8", "seaborn>=0.13"\n]\n\n[project.optional-dependencies]\ndev = ["pytest>=8", "jupyter>=1.0"]\ndemo = ["streamlit>=1.42"]\n\n[tool.setuptools.packages.find]\ninclude = ["src*"]\n\n[tool.pytest.ini_options]\npythonpath = ["."]\ntestpaths = ["tests"]\n',
}
from pathlib import Path
import os, sys

PROJECT_ROOT = Path("/content/quantcomp")
for relative_path, contents in project_sources.items():
    destination = PROJECT_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(contents)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
(PROJECT_ROOT / "data").mkdir(exist_ok=True)
(PROJECT_ROOT / "results").mkdir(exist_ok=True)
print(f"Project ready at {PROJECT_ROOT}")


## 2. Upload the dataset

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()  # Choose Symptom2Disease.csv from your computer.
csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_names:
    raise FileNotFoundError("Upload the Symptom2Disease.csv file to continue.")
DATA_PATH = PROJECT_ROOT / "data" / "Symptom2Disease.csv"
shutil.move(csv_names[0], DATA_PATH)
print("Using:", DATA_PATH)


## 3. Check the data and run the project tests

In [ ]:
from src.data import LABELS, load_frame, make_split

frame = load_frame(DATA_PATH)
print(f"Selected unique examples: {len(frame)}")
print("Classes:", frame.label.value_counts().to_dict())
split0 = make_split(seed=0, path=DATA_PATH)
print(f"Seed 0 split: {len(split0.y_train)} train / {len(split0.y_test)} test")
assert len(frame) == 388
assert len(split0.y_train) == 291 and len(split0.y_test) == 97


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT_ROOT, check=True)


## 4. Run classical baselines (five splits)

In [ ]:
import json
import numpy as np
from src.runner import run_classical

classical_rows = run_classical(seeds=range(5), path=DATA_PATH)
classical_summary = {}
for name in sorted({row["model"] for row in classical_rows}):
    subset = [row for row in classical_rows if row["model"] == name]
    scores = [row["accuracy"] for row in subset]
    parameter_counts = [row["parameters"] for row in subset if row["parameters"] is not None]
    classical_summary[name] = {
        "accuracy_mean": float(np.mean(scores)),
        "accuracy_std": float(np.std(scores, ddof=1)),
        "parameters_mean": float(np.mean(parameter_counts)) if parameter_counts else None,
    }
(PROJECT_ROOT / "results" / "classical.json").write_text(json.dumps(classical_rows, indent=2))
(PROJECT_ROOT / "results" / "classical_summary.json").write_text(json.dumps(classical_summary, indent=2))
for name, result in classical_summary.items():
    print(f"{name:18s} {result['accuracy_mean']:.1%} ± {result['accuracy_std']:.1%}")


## 5. Phase A: tune circuit structure

This runs all 19 configurations on the validation fold for seed 0 and saves each completed run under `results/tuning_A.json`. It uses five epochs, matching the project experiment. This is the longest notebook section.


In [ ]:
from src.tuning import phase_a

phase_a(seeds=(0,), epochs=5, output=PROJECT_ROOT / "results" / "tuning_A.json", path=DATA_PATH)
print("Phase A complete; saved results/tuning_A.json")


## 6. Select the best validation configuration for each encoding

In [ ]:
from src.runner import select_best

best_configs = select_best(PROJECT_ROOT / "results" / "tuning_A.json")
for encoding, config in best_configs.items():
    print(encoding, config)


## 7. Final QNN evaluation on five test splits

Each selected configuration is trained on the full training split for five epochs, then evaluated once on that split’s test data. The test labels are not used to select configurations.


In [ ]:
from src.runner import final_evaluation

qnn_rows = final_evaluation(best=best_configs, seeds=range(5), epochs=5, path=DATA_PATH)
print("Completed", len(qnn_rows), "encoding/split evaluations")


In [ ]:
qnn_summary = json.loads((PROJECT_ROOT / "results" / "qnn_summary.json").read_text())
for encoding, result in qnn_summary.items():
    print(f"{encoding:10s} {result['mean_accuracy']:.1%} ± {result['std_accuracy']:.1%}")


## 8. View generated plots

In [ ]:
from IPython.display import display, Image

plot_names = [
    "accuracy_by_encoding.png", "accuracy_by_depth.png", "training_curves.png",
    "confusion_angle.png", "confusion_zz.png", "confusion_amplitude.png",
]
for plot_name in plot_names:
    plot_path = PROJECT_ROOT / "results" / plot_name
    print(plot_name)
    display(Image(filename=str(plot_path)))


## 9. Train and try the live sentence demo

In [ ]:
from src.runner import save_demo_model
from src.qnn import QNNConfig
from src.predict import predict

# The demo artifact uses the strongest validation configuration from Phase A.
demo_config = max(
    json.loads((PROJECT_ROOT / "results" / "tuning_A.json").read_text()),
    key=lambda row: row["validation_accuracy"],
)["config"]
demo_model_path = save_demo_model(config=QNNConfig(**demo_config), seed=0, epochs=5, path=DATA_PATH)
print("Saved demo model:", demo_model_path)


In [ ]:
sentence = "I have a high fever, chills and I keep sweating at night"  # Edit this sentence and rerun.
predicted_label, probabilities = predict(sentence)
print("Predicted dataset label:", predicted_label)
for label, probability in sorted(probabilities.items(), key=lambda item: -item[1]):
    print(f"{label:20s} {probability:.1%}")


## 10. Download the generated results

In [ ]:
import shutil
from google.colab import files

results_zip = PROJECT_ROOT / "results_colab.zip"
shutil.make_archive(str(results_zip.with_suffix("")), "zip", PROJECT_ROOT / "results")
files.download(str(results_zip))


## Interpretation and limitations

- TF-IDF, SVD, feature scaling, optimization, and final argmax are classical; only encoding, variational gates, and measurement are quantum.
- Classical models are expected to win. The results from this dataset show no quantum advantage or speedup.
- The 8-dimensional SVD representation discards substantial text information, and each test split contains only 97 sentences.
- Amplitude encoding normalizes the input vector and therefore discards its length.
- The simulator returns exact probabilities by default; this notebook does not model finite-shot hardware noise.
- Do not use predictions for medical decisions.
